# NB171: The Cross-Level Factor

## Target: Derive (p₃/p₂)² = 25/9 in x_q = (4/7)(25/9)

NB170 confirmed x_q = 100/63 and factored it as x(R₀) × cross-level = (4/7)(25/9).
The base x(R₀) = 4/7 is derived (NB161). The cross-level factor 25/9 is **measured** — this notebook aims to derive it.

### Strategy

1. **Per-level amplification anatomy** — decompose x_q level by level
2. **Subsystem cascades** — run {2,3}, {2,3,5}, {2,3,5,7} and watch x_q build
3. **Linearized steady-state** — solve the cascade ODE analytically for small ε
4. **Transfer function** — the ratio of driven amplitudes between levels determines the cross-level factor
5. **Connect to (p₃/p₂)²** — show why the cascade architecture produces this specific ratio

In [3]:
# ── Setup ──
import sys, numpy as np
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import (SA, RHO, KAPPA, EPSILON, OMEGA,
                               PHYSICAL_CROSSINGS, CP_PAIRS, SM_TARGETS)
from solenoid_system import SolenoidSystem

print(f"P_4 = {SA.P}, phi(P_4) = {SA.PHI}, rho = kappa = eps = 1/sqrt({SA.P}) = {RHO:.6f}")
print(f"Primes: {SA.primes}")
print(f"omega = {OMEGA:.4f}")
print(f"Physical crossings: {PHYSICAL_CROSSINGS}")
print(f"CP pairs: QUARK={CP_PAIRS['QUARK']}, LEPTON={CP_PAIRS['LEPTON']}")

P_4 = 210, phi(P_4) = 48, rho = kappa = eps = 1/sqrt(210) = 0.069007
Primes: [2, 3, 5, 7]
omega = 6.2832
Physical crossings: {'QUARK_g1': {'ci': 11, 'a3': 1, 'a5': 0, 'a7s': 4, 'type': 'QUARK', 'gen': 'g1'}, 'LEPTON_g1': {'ci': 31, 'a3': 0, 'a5': 0, 'a7s': 1, 'type': 'LEPTON', 'gen': 'g1'}, 'LEPTON_g2': {'ci': 61, 'a3': 0, 'a5': 0, 'a7s': 5, 'type': 'LEPTON', 'gen': 'g2'}, 'QUARK_g2': {'ci': 191, 'a3': 1, 'a5': 0, 'a7s': 2, 'type': 'QUARK', 'gen': 'g2'}}
CP pairs: QUARK=(1, 4, 2), LEPTON=(0, 1, 5)


## Section 1: Per-Level Amplification Anatomy

NB170 measured x at each covering level: R₀ = 0.5714, R₁ = 0.7351, R₂ = 0.8132, R₃ = 1.5866.

The **step ratios** are:
- R₀→R₁: x₁/x₀ 
- R₁→R₂: x₂/x₁ 
- R₂→R₃: x₃/x₂ 

The product of all step ratios must equal the cross-level factor 25/9.

Question: do the step ratios have prime structure? Can each one be expressed as a simple function of {2,3,5,7}?

In [6]:
# ── Section 1: Per-level exponent decomposition ──
# Use window-0 (T=211, evaluate at coprime crossing indices) — NB170 showed T-independence

from solenoid_jax import integrate_all_branches_jax, warmup
warmup()

sys0 = SolenoidSystem()
all_branches = sys0.all_branches()
cis = SA.coprime_indices(SA.P)
a3, a5, a7 = SA.sector_labels(cis)
T_max = 211

t_eval = cis.astype(float)
res = integrate_all_branches_jax(all_branches, t_eval, float(T_max))
print(f"Integrated {len(res)} branches at {len(cis)} coprime crossings, T={T_max}")

# Compute per-level RMS at each coprime crossing
rms = np.zeros((len(cis), 4))
for idx in range(len(cis)):
    for k in range(4):
        Rk = np.array([res[br][idx, k] for br in all_branches])
        Rk_w = np.mod(Rk, 2 * np.pi)
        Rk_w[Rk_w > np.pi] -= 2 * np.pi
        rms[idx, k] = np.sqrt(np.mean(Rk_w**2))

# Quark CP ratio at each level
ch_a3, a7_g1, a7_g2 = CP_PAIRS['QUARK']
g1_idx = np.where((a3 == ch_a3) & (a5 == 0) & (a7 == a7_g1))[0][0]
g2_idx = np.where((a3 == ch_a3) & (a5 == 0) & (a7 == a7_g2))[0][0]

ms_md_pdg = SM_TARGETS['m_s/m_d'][0]  # central value

print(f"\nQuark g1 crossing index: ci={cis[g1_idx]}, g2: ci={cis[g2_idx]}")
print(f"\n{'Level':<8} {'RMS_g1':<12} {'RMS_g2':<12} {'CP_ratio':<14} {'x_k':<12}")
print('-' * 60)

cp_q = {}
x_levels = []
for k in range(4):
    r_g1 = rms[g1_idx, k]
    r_g2 = rms[g2_idx, k]
    cp = r_g1 / r_g2
    cp_q[k] = cp
    x_k = np.log(ms_md_pdg) / np.log(cp) if cp > 1 else np.nan
    x_levels.append(x_k)
    print(f"R{k:<7d} {r_g1:<12.6f} {r_g2:<12.6f} {cp:<14.8f} {x_k:<12.6f}")

# Step ratios
print(f"\n{'='*60}")
print(f"STEP RATIOS (per-level amplification)")
print(f"{'='*60}")
step_ratios = []
for k in range(1, 4):
    sr = x_levels[k] / x_levels[k-1]
    step_ratios.append(sr)
    print(f"  x(R{k})/x(R{k-1}) = {x_levels[k]:.6f} / {x_levels[k-1]:.6f} = {sr:.8f}")

cross_level = x_levels[3] / x_levels[0]
print(f"\n  Total cross-level = x(R3)/x(R0) = {cross_level:.8f}")
print(f"  Target 25/9 = {25/9:.8f}")
print(f"  Deviation: {(cross_level - 25/9)/(25/9)*1e6:.0f} ppm")
print(f"\n  Product of step ratios: {np.prod(step_ratios):.8f}")
print(f"  (Consistency: {abs(np.prod(step_ratios) - cross_level):.2e})")

  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 1.21s
Integrated 210 branches at 48 coprime crossings, T=211

Quark g1 crossing index: ci=11, g2: ci=191

Level    RMS_g1       RMS_g2       CP_ratio       x_k         
------------------------------------------------------------
R0       2.075590     0.010975     189.11186781   0.571450    
R1       1.618601     0.027498     58.86346488    0.735109    
R2       1.737103     0.043644     39.80144226    0.813195    
R3       1.846494     0.279486     6.60674225     1.586646    

STEP RATIOS (per-level amplification)
  x(R1)/x(R0) = 0.735109 / 0.571450 = 1.28639385
  x(R2)/x(R1) = 0.813195 / 0.735109 = 1.10622360
  x(R3)/x(R2) = 1.586646 / 0.813195 = 1.95112618

  Total cross-level = x(R3)/x(R0) = 2.77652911
  Target 25/9 = 2.77777778
  Deviation: -450 ppm

  Product of step ratios: 2.77652911
  (Consistency: 4.44e-16)


## Section 2: Prime Arithmetic of Step Ratios

The step ratios are:
- R₀→R₁: 1.28639 
- R₁→R₂: 1.10622
- R₂→R₃: 1.95113

Do these factor into prime expressions? The total product is 25/9 = p₃²/p₂². 

Key observation: the cascade couples level k to level k-1 via the prime pₖ. The coupling terms in the cascade ODE involve 1/pₖ. So the **transition from level k-1 to k is determined by pₖ**. This means:

- R₀→R₁: determined by p₁ = 2 (coupling between levels 0 and 1)
- R₁→R₂: determined by p₂ = 3
- R₂→R₃: determined by p₃ = 5 (and p₄ = 7 is the outermost)

Let's search for simple prime expressions that match, and also examine the **log-CP structure** rather than the exponent ratios directly.

In [7]:
# ── Section 2: Search for prime structure in step ratios ──
from itertools import product as iterproduct

primes = SA.primes  # [2, 3, 5, 7]
step = {
    '0→1': step_ratios[0],
    '1→2': step_ratios[1],
    '2→3': step_ratios[2],
}

print("STEP RATIO PRIME SEARCH")
print("=" * 70)

# Search for a/b where a,b are products of small powers of {2,3,5,7}
# with small exponents e in [-3, 3]
best_matches = {}
for name, val in step.items():
    candidates = []
    for e2, e3, e5, e7 in iterproduct(range(-3, 4), repeat=4):
        frac_val = (2**e2) * (3**e3) * (5**e5) * (7**e7)
        if frac_val > 0.5 and frac_val < 5.0:
            ppm = abs(val - frac_val) / val * 1e6
            if ppm < 5000:  # within 0.5%
                candidates.append((ppm, frac_val, e2, e3, e5, e7))
    candidates.sort()
    best_matches[name] = candidates[:5]
    print(f"\n{name} = {val:.8f}")
    for ppm, fv, e2, e3, e5, e7 in candidates[:5]:
        parts = []
        for e, p in [(e2, 2), (e3, 3), (e5, 5), (e7, 7)]:
            if e != 0:
                parts.append(f"{p}^{e}" if e != 1 else str(p))
        expr = " × ".join(parts) if parts else "1"
        frac = Fraction(fv).limit_denominator(1000)
        print(f"  {frac} = {expr} = {fv:.8f}  ({ppm:.0f} ppm)")

# Also look at the LOGARITHMS of CP ratios
print(f"\n\n{'='*70}")
print("LOG-CP STRUCTURE")
print(f"{'='*70}")
for k in range(4):
    ln_cp = np.log(cp_q[k])
    print(f"  ln(CP_q[R{k}]) = {ln_cp:.8f}")

print(f"\n  Ratios of ln(CP):")
for k in range(1, 4):
    ratio = np.log(cp_q[k-1]) / np.log(cp_q[k])
    print(f"  ln(CP[R{k-1}]) / ln(CP[R{k}]) = {ratio:.8f}")

# The exponent x_k = ln(target)/ln(CP_k), so x_{k-1}/x_k = ln(CP_k)/ln(CP_{k-1})
# i.e., step_ratio_k = x_k/x_{k-1} = ln(CP_{k-1})/ln(CP_k)
print(f"\n  Cross-check: ln(CP[R0])/ln(CP[R3]) = {np.log(cp_q[0])/np.log(cp_q[3]):.8f}")
print(f"  This should equal cross-level = {cross_level:.8f}")

STEP RATIO PRIME SEARCH

0→1 = 1.28639385
  9/7 = 3^2 × 7^-1 = 1.28571429  (528 ppm)

1→2 = 1.10622360
  54/49 = 2 × 3^3 × 7^-2 = 1.10204082  (3781 ppm)
  10/9 = 2 × 3^-2 × 5 = 1.11111111  (4418 ppm)

2→3 = 1.95112618
  35/18 = 2^-1 × 3^-2 × 5 × 7 = 1.94444444  (3425 ppm)
  49/25 = 5^-2 × 7^2 = 1.96000000  (4548 ppm)


LOG-CP STRUCTURE
  ln(CP_q[R0]) = 5.24233873
  ln(CP_q[R1]) = 4.07522061
  ln(CP_q[R2]) = 3.68390315
  ln(CP_q[R3]) = 1.88809068

  Ratios of ln(CP):
  ln(CP[R0]) / ln(CP[R1]) = 1.28639385
  ln(CP[R1]) / ln(CP[R2]) = 1.10622360
  ln(CP[R2]) / ln(CP[R3]) = 1.95112618

  Cross-check: ln(CP[R0])/ln(CP[R3]) = 2.77652911
  This should equal cross-level = 2.77652911


## Section 3: Subsystem Cascades

The total cross-level factor 25/9 = (p₃/p₂)² involves exactly two primes. The cascade has four levels indexed by primes {2,3,5,7}. 

**Key idea**: Run truncated cascades with subsets of primes to see how the cross-level factor builds:
- 2-prime: {2,3} — gives only R₀,R₁ 
- 3-prime: {2,3,5} — gives R₀,R₁,R₂
- 4-prime: {2,3,5,7} — the full system

If the cross-level factor is a local property of the cascade architecture, it should be computable from the ODE structure alone.

**Also**: what happens with different 4-prime sets? E.g., {2,3,5,11}? This tests whether 25/9 depends on specific prime VALUES or only the cascade POSITION.

In [9]:
# ── Section 3: Subsystem cascades ──
# Run cascades with different prime subsets and measure cross-level factor

from math import gcd
from functools import reduce
from scipy.integrate import solve_ivp
from itertools import product as iterproduct

def run_subsystem(primes_sub, verbose=True):
    """Run a cascade with a subset of primes and compute per-level exponents."""
    P = reduce(lambda a, b: a * b, primes_sub)
    phi_P = reduce(lambda a, b: a * b, [p - 1 for p in primes_sub])
    kappa_sub = 1.0 / np.sqrt(P)
    eps_sub = kappa_sub
    
    # Coprime crossings
    cis_sub = np.array([k for k in range(1, P + 1) if gcd(k, P) == 1])
    
    # Create system and branches
    ss = SolenoidSystem(primes=primes_sub, kappa=kappa_sub, epsilon=eps_sub)
    n_levels = len(primes_sub)
    T_max = float(P + 1)  # one full period
    
    # All branches
    all_br = list(iterproduct(*[range(p) for p in primes_sub]))
    
    # Integrate using scipy
    t_eval = cis_sub.astype(float)
    
    results = {}
    for br in all_br:
        R0 = ss.initial_R(br)
        sol = solve_ivp(ss.cascade_ode, [0, T_max + 1], R0, method='DOP853',
                       t_eval=t_eval, rtol=1e-12, atol=1e-14)
        results[br] = sol.y.T  # (n_crossings, n_levels)
    
    # Compute RMS at each crossing and level
    rms_sub = np.zeros((len(cis_sub), n_levels))
    for idx in range(len(cis_sub)):
        for k in range(n_levels):
            Rk = np.array([results[br][idx, k] for br in all_br])
            Rk_w = np.mod(Rk, 2 * np.pi)
            Rk_w[Rk_w > np.pi] -= 2 * np.pi
            rms_sub[idx, k] = np.sqrt(np.mean(Rk_w**2))
    
    # CP ratio: use pair of crossings that maximize the CP spread at outermost level
    best_cp = 0
    best_pair = (0, 1)
    for i in range(len(cis_sub)):
        for j in range(i+1, len(cis_sub)):
            if rms_sub[j, -1] > 0:
                cp = rms_sub[i, -1] / rms_sub[j, -1]
                if cp > best_cp:
                    best_cp = cp
                    best_pair = (i, j)
            if rms_sub[i, -1] > 0:
                cp_inv = rms_sub[j, -1] / rms_sub[i, -1]
                if cp_inv > best_cp:
                    best_cp = cp_inv
                    best_pair = (j, i)
    
    g1_idx, g2_idx = best_pair
    
    if verbose:
        print(f"\nPrimes: {primes_sub}, P={P}, phi={phi_P}, crossings={len(cis_sub)}")
        print(f"Best CP pair: ci={cis_sub[g1_idx]} / ci={cis_sub[g2_idx]}")
    
    # Per-level CP and x
    cp_ratios = []
    ln_cps = []
    for k in range(n_levels):
        cp = rms_sub[g1_idx, k] / rms_sub[g2_idx, k] if rms_sub[g2_idx, k] > 0 else 1.0
        cp_ratios.append(cp)
        ln_cps.append(np.log(cp) if cp > 1 else np.nan)
    
    # Cross-level factor
    if not np.isnan(ln_cps[0]) and not np.isnan(ln_cps[-1]) and ln_cps[-1] != 0:
        cross = ln_cps[0] / ln_cps[-1]
    else:
        cross = np.nan
    
    if verbose:
        for k in range(n_levels):
            print(f"  R{k}: CP={cp_ratios[k]:.6f}, ln(CP)={ln_cps[k]:.6f}")
        print(f"  Cross-level = ln(CP₀)/ln(CP_{n_levels-1}) = {cross:.6f}")
    
    return {
        'primes': primes_sub, 'P': P, 'rms': rms_sub, 'cis': cis_sub,
        'cp_ratios': cp_ratios, 'ln_cps': ln_cps, 'cross_level': cross,
        'g1_idx': g1_idx, 'g2_idx': g2_idx,
    }

# Run subsystems
print("SUBSYSTEM CASCADE ANALYSIS")
print("=" * 70)
r2 = run_subsystem([2, 3])
r3 = run_subsystem([2, 3, 5])
r4 = run_subsystem([2, 3, 5, 7])

# Also test with different outermost primes
print("\n\nALTERNATIVE PRIME SETS (same structure, different values)")
print("=" * 70)
r4b = run_subsystem([2, 3, 5, 11])
r4c = run_subsystem([2, 3, 5, 13])
r4d = run_subsystem([2, 3, 7, 11])

print("\n\nSUMMARY: CROSS-LEVEL FACTORS")
print("=" * 70)
print(f"{'Primes':<20} {'Cross-level':<14}")
print("-" * 40)
for label, r in [
    ("{2,3}", r2),
    ("{2,3,5}", r3),
    ("{2,3,5,7}", r4),
    ("{2,3,5,11}", r4b),
    ("{2,3,5,13}", r4c),
    ("{2,3,7,11}", r4d),
]:
    print(f"{label:<20} {r['cross_level']:<14.6f}")

SUBSYSTEM CASCADE ANALYSIS

Primes: [2, 3], P=6, phi=2, crossings=2
Best CP pair: ci=1 / ci=5
  R0: CP=2.793295, ln(CP)=1.027222
  R1: CP=1.113616, ln(CP)=0.107612
  Cross-level = ln(CP₀)/ln(CP_1) = 9.545563

Primes: [2, 3, 5], P=30, phi=8, crossings=8
Best CP pair: ci=11 / ci=29
  R0: CP=28.217767, ln(CP)=3.339952
  R1: CP=10.133887, ln(CP)=2.315885
  R2: CP=27.477505, ln(CP)=3.313368
  Cross-level = ln(CP₀)/ln(CP_2) = 1.008023

Primes: [2, 3, 5, 7], P=210, phi=48, crossings=48
Best CP pair: ci=37 / ci=83
  R0: CP=33.039475, ln(CP)=3.497703
  R1: CP=11.520405, ln(CP)=2.444120
  R2: CP=21.031911, ln(CP)=3.046041
  R3: CP=39.101910, ln(CP)=3.666171
  Cross-level = ln(CP₀)/ln(CP_3) = 0.954048


ALTERNATIVE PRIME SETS (same structure, different values)

Primes: [2, 3, 5, 11], P=330, phi=80, crossings=80
Best CP pair: ci=53 / ci=307
  R0: CP=26.759294, ln(CP)=3.286882
  R1: CP=32.594062, ln(CP)=3.484130
  R2: CP=41.199734, ln(CP)=3.718432
  R3: CP=48.523140, ln(CP)=3.882041
  Cross-level =

In [14]:
# Compact summary of quark channel structure
ms_md = SM_TARGETS['m_s/m_d'][0]  # 20.0
x_levels = [np.log(ms_md) / np.log(cp_q[k]) for k in range(4)]

print("QUARK CHANNEL CP & EXPONENT STRUCTURE")
print("=" * 70)
print(f"{'Level':<8} {'CP ratio':<14} {'ln(CP)':<14} {'x_k':<14} {'Step ratio'}")
print("-" * 70)
for k in range(4):
    sr_str = f"{step_ratios[k]:.6f}" if k < len(step_ratios) else "—"
    print(f"R{k:<6} {cp_q[k]:<14.8f} {np.log(cp_q[k]):<14.8f} {x_levels[k]:<14.8f} {sr_str}")

print(f"\nCross-level = x(R3)/x(R0) = {cross_level:.8f}")
print(f"25/9 = {25/9:.8f}")
print(f"Dev = {(cross_level - 25/9)/(25/9)*1e6:+.0f} ppm")

# Extract crossing indices from PHYSICAL_CROSSINGS
print(f"\nPHYSICAL_CROSSINGS structure: {PHYSICAL_CROSSINGS}")
ci_g1 = PHYSICAL_CROSSINGS['QUARK_g1']['ci']
ci_g2 = PHYSICAL_CROSSINGS['QUARK_g2']['ci']

print(f"\nQUARK CROSSINGS:")
print(f"  g1 = ci={ci_g1} (early, transient-dominated)")
print(f"  g2 = ci={ci_g2} (late, steady-state dominated)")
print(f"  Decay at g1: e^(-κ·{ci_g1}) = {np.exp(-KAPPA*ci_g1):.4f}")
print(f"  Decay at g2: e^(-κ·{ci_g2}) = {np.exp(-KAPPA*ci_g2):.2e}")
print(f"\n  → g1 retains {np.exp(-KAPPA*ci_g1)*100:.1f}% of transient")
print(f"  → g2 ~0% (steady-state only)")
print(f"  → CP asymmetry = transient spread / steady-state level")

# Step ratios and coupling primes
print(f"\nSTEP RATIOS (= ln(CP_{{k-1}})/ln(CP_k)):")
for k in range(3):
    r = np.log(cp_q[k]) / np.log(cp_q[k+1])
    p_coupling = SA.primes[k+1]  # prime that couples level k to k+1
    print(f"  R{k}→R{k+1}: {r:.6f}  (coupling prime p={p_coupling})")

QUARK CHANNEL CP & EXPONENT STRUCTURE
Level    CP ratio       ln(CP)         x_k            Step ratio
----------------------------------------------------------------------
R0      189.11186781   5.24233873     0.57144958     1.286394
R1      58.86346488    4.07522061     0.73510923     1.106224
R2      39.80144226    3.68390315     0.81319518     1.951126
R3      6.60674225     1.88809068     1.58664640     —

Cross-level = x(R3)/x(R0) = 2.77652911
25/9 = 2.77777778
Dev = -450 ppm

PHYSICAL_CROSSINGS structure: {'QUARK_g1': {'ci': 11, 'a3': 1, 'a5': 0, 'a7s': 4, 'type': 'QUARK', 'gen': 'g1'}, 'LEPTON_g1': {'ci': 31, 'a3': 0, 'a5': 0, 'a7s': 1, 'type': 'LEPTON', 'gen': 'g1'}, 'LEPTON_g2': {'ci': 61, 'a3': 0, 'a5': 0, 'a7s': 5, 'type': 'LEPTON', 'gen': 'g2'}, 'QUARK_g2': {'ci': 191, 'a3': 1, 'a5': 0, 'a7s': 2, 'type': 'QUARK', 'gen': 'g2'}}

QUARK CROSSINGS:
  g1 = ci=11 (early, transient-dominated)
  g2 = ci=191 (late, steady-state dominated)
  Decay at g1: e^(-κ·11) = 0.4681
  Decay 

In [ ]:
## Section 4: Transient/Steady-State Decomposition

The subsystem cascade analysis (Section 3) was wrong — it picked generic crossing pairs, not the quark-specific CRT pair. **The cross-level factor 25/9 is specific to the quark channel crossings (ci=11, ci=191).**

The right approach: at ci=191 (g2), the transient has decayed to ~0 (e^{-κ·191} = 1.9×10⁻⁶). So the RMS at g2 IS the steady-state amplitude. At ci=11 (g1), the transient retains 47% of its initial value.

**The CP ratio at each level** = (transient + steady-state at g1) / (steady-state at g2)

The cross-level factor measures how differently each level filters the transient relative to the steady state. If we decompose:

$$\text{CP}_k = \frac{\text{RMS}_{g1}(k)}{\text{RMS}_{g2}(k)} = \frac{\sqrt{\sigma^2_{\text{trans}}(k) + A^2_{\text{SS}}(k, t_{g1})}}{|A_{\text{SS}}(k, t_{g2})|}$$

where σ²_trans is the branch-dependent variance and A_SS is the branch-independent steady-state value.

In [18]:
# ── Section 4: Correct Transient/SS Decomposition using JAX data (res) ──
# Section 1 used 'res' (JAX), 'cis' (coprime indices), and CRT-based g1/g2 indices.
# Section 3 had 'results' from subsystem runs — DO NOT use for decomposition.

# Reuse Section 1 variables
n_levels = 4
all_br = list(res.keys())
n_branches = len(all_br)  # 210

# g1_idx, g2_idx from Section 1 (CRT-based, unique with a5=0 constraint)
# g1_idx points to ci=11, g2_idx points to ci=191
ci_g1_actual = cis[g1_idx]
ci_g2_actual = cis[g2_idx]
print(f"Quark crossings: g1 → ci={ci_g1_actual}, g2 → ci={ci_g2_actual}")
print(f"Branches: {n_branches}\n")

print("TRANSIENT/STEADY-STATE DECOMPOSITION (correct JAX data)")
print("=" * 80)
print(f"{'Level':<6} {'RMS_g1':<12} {'RMS_g2':<12} {'CP':<14} {'mean_g2':<12} {'std_g1':<12} {'std_g2':<12}")
print("-" * 80)

rms_g1_c = np.zeros(n_levels)
rms_g2_c = np.zeros(n_levels)
mean_g1_c = np.zeros(n_levels)
mean_g2_c = np.zeros(n_levels)
std_g1_c = np.zeros(n_levels)
std_g2_c = np.zeros(n_levels)

for k in range(n_levels):
    vals_g1 = np.array([res[br][g1_idx, k] for br in all_br])
    vals_g2 = np.array([res[br][g2_idx, k] for br in all_br])
    
    # Wrap to [-pi, pi]
    def wrap(x):
        x = np.mod(x, 2*np.pi)
        x[x > np.pi] -= 2*np.pi
        return x
    
    v_g1 = wrap(vals_g1.copy())
    v_g2 = wrap(vals_g2.copy())
    
    rms_g1_c[k] = np.sqrt(np.mean(v_g1**2))
    rms_g2_c[k] = np.sqrt(np.mean(v_g2**2))
    mean_g1_c[k] = np.mean(v_g1)
    mean_g2_c[k] = np.mean(v_g2)
    std_g1_c[k] = np.std(v_g1)
    std_g2_c[k] = np.std(v_g2)
    
    cp = rms_g1_c[k] / rms_g2_c[k]
    print(f"R{k:<4} {rms_g1_c[k]:<12.6f} {rms_g2_c[k]:<12.6f} {cp:<14.8f} "
          f"{mean_g2_c[k]:<+12.6f} {std_g1_c[k]:<12.6f} {std_g2_c[k]:<12.6f}")

print(f"\nCross-check CP(R3): {rms_g1_c[3]/rms_g2_c[3]:.8f} vs Section 1: {cp_q[3]:.8f}")

# Verify: are these the same as Section 1?
print(f"\nConsistency with Section 1 CP values:")
for k in range(4):
    cp_here = rms_g1_c[k] / rms_g2_c[k]
    print(f"  R{k}: Section 1 = {cp_q[k]:.6f}, here = {cp_here:.6f}, match = {abs(cp_here - cp_q[k]) < 1e-6}")

Quark crossings: g1 → ci=11, g2 → ci=191
Branches: 210

TRANSIENT/STEADY-STATE DECOMPOSITION (correct JAX data)
Level  RMS_g1       RMS_g2       CP             mean_g2      std_g1       std_g2      
--------------------------------------------------------------------------------
R0    2.075590     0.010975     189.11186781   -0.010975    1.470581     0.000006    
R1    1.618601     0.027498     58.86346488    +0.027498    1.574795     0.000040    
R2    1.737103     0.043644     39.80144226    -0.043644    1.736365     0.000096    
R3    1.846494     0.279486     6.60674225     +0.279486    1.837814     0.000111    

Cross-check CP(R3): 6.60674225 vs Section 1: 6.60674225

Consistency with Section 1 CP values:
  R0: Section 1 = 189.111868, here = 189.111868, match = True
  R1: Section 1 = 58.863465, here = 58.863465, match = True
  R2: Section 1 = 39.801442, here = 39.801442, match = True
  R3: Section 1 = 6.606742, here = 6.606742, match = True


In [17]:
# ── Analytical R₀ solution ──
# R₀ ODE: dR₀/dt = ε·sin(ωt) − κ·R₀  (DECOUPLED — does not depend on R₁,R₂,R₃)
# Exact solution at integer t=n:
# R₀(n) = (2πj₀ + α)·e^{−κn} − α,  where α = ε·ω/(ω²+κ²)

alpha = EPSILON * OMEGA / (OMEGA**2 + KAPPA**2)
print(f"α = ε·ω/(ω²+κ²) = {alpha:.10f}")

def R0_analytical(n, j0):
    C0 = 2*np.pi*j0 + alpha
    return C0 * np.exp(-KAPPA*n) - alpha

# Verify with standalone scipy integration of R₀ ALONE
from scipy.integrate import solve_ivp

def r0_ode_standalone(t, y):
    return [EPSILON * np.sin(OMEGA * t) - KAPPA * y[0]]

# Branch j₀=1
sol_standalone = solve_ivp(r0_ode_standalone, [0, 211], [2*np.pi],
                           t_eval=[11.0, 191.0], method='DOP853',
                           rtol=1e-14, atol=1e-16)
print(f"\nSTANDALONE R₀ integration (j₀=1):")
print(f"  R₀(11) = {sol_standalone.y[0,0]:.10f}  (analytical: {R0_analytical(11, 1):.10f})")
print(f"  R₀(191) = {sol_standalone.y[0,1]:.10f}  (analytical: {R0_analytical(191, 1):.10f})")

# Full cascade integration of branch (1,0,0,0)
sol_full = solve_ivp(sys0.cascade_ode, [0, 212], sys0.initial_R((1,0,0,0)),
                     t_eval=[11.0, 191.0], method='DOP853',
                     rtol=1e-14, atol=1e-16)
print(f"\nFULL CASCADE integration of (1,0,0,0):")
print(f"  R₀(11) = {sol_full.y[0,0]:.10f}  R₁(11) = {sol_full.y[1,0]:.10f}")
print(f"  R₀(191) = {sol_full.y[0,1]:.10f}  R₁(191) = {sol_full.y[1,1]:.10f}")

# JAX results from Section 1 (variable: res)
print(f"\nJAX results (res) for branch (1,0,0,0):")
jax_g1 = res[(1,0,0,0)][g1_ci_idx]  # using Section 1 g1_ci_idx based on coprime_cis
jax_g2 = res[(1,0,0,0)][g2_ci_idx]
print(f"  R₀ at g1 idx: {jax_g1[0]:.10f}")
print(f"  R₀ at g2 idx: {jax_g2[0]:.10f}")

# Check: did Section 1 and Section 4 use the SAME variable?
print(f"\n  Section 1 variable 'res': {id(res)}")
print(f"  Section 4 variable 'results': {id(results)}")
print(f"  Are they the same object? {res is results}")

α = ε·ω/(ω²+κ²) = 0.0109814099


c:\Users\mlf\.conda\envs\concentric\Lib\site-packages\scipy\integrate\_ivp\rk.py:505: UserWarning: At least one element of `rtol` is too small. Setting `rtol = np.maximum(rtol, 2.220446049250313e-14)`.
  super().__init__(fun, t0, y0, t_bound, max_step, rtol, atol,



STANDALONE R₀ integration (j₀=1):
  R₀(11) = 2.9353216113  (analytical: 2.9353216113)
  R₀(191) = -0.0109695296  (analytical: -0.0109695296)

FULL CASCADE integration of (1,0,0,0):
  R₀(11) = 2.9353216113  R₁(11) = 1.1117872298
  R₀(191) = -0.0109695296  R₁(191) = 0.0275246936

JAX results (res) for branch (1,0,0,0):
  R₀ at g1 idx: 2.9353216119
  R₀ at g2 idx: -0.0109695296

  Section 1 variable 'res': 2177451146624
  Section 4 variable 'results': 2175869715520
  Are they the same object? False


## Section 5: Steady-State Scaling — The Key to 25/9

The decomposition reveals the mechanism. The CP ratio is:

$$\text{CP}_k = \frac{\text{RMS}_{\text{transient}}(k)}{\text{RMS}_{\text{steady-state}}(k)}$$

The transient (g1) barely changes across levels: 2.08 → 1.85 (factor 0.89). But the steady-state (g2) changes dramatically: 0.011 → 0.280 (factor 25.5).

**The cross-level factor 25/9 is dominated by the steady-state amplitude scaling.**

The analytical steady state at R₀ is known:
$$R_0^{SS} = -\alpha = -\frac{\varepsilon\omega}{\omega^2 + \kappa^2} \approx -\frac{1}{2\pi\sqrt{P_4}}$$

The question: what determines the steady-state at higher levels? If the ratios of |SS(Rₖ)/SS(R₀)| have prime structure, we can derive 25/9.

In [21]:
# ── Section 5: Steady-State Amplitude Scaling ──
# At ci=191 (g2), transient is dead. mean_g2 IS the steady-state value.

ss_vals = mean_g2_c.copy()  # Steady-state R_k at ci=191
ss_abs = np.abs(ss_vals)

# Analytical R₀ steady state
R0_SS_analytical = -alpha
print("STEADY-STATE AMPLITUDES at ci=191")
print("=" * 70)
print(f"R₀: {ss_vals[0]:+.8f}  |R₀| = {ss_abs[0]:.8f}  (analytical: {abs(R0_SS_analytical):.8f})")
for k in range(1, 4):
    print(f"R{k}: {ss_vals[k]:+.8f}  |R{k}| = {ss_abs[k]:.8f}")

print(f"\nSIGN PATTERN: {['+' if v > 0 else '-' for v in ss_vals]}")
print(f"  → Alternating: {all(ss_vals[i]*ss_vals[i+1] < 0 for i in range(3))}")

# Ratios relative to R₀
print(f"\nSCALING RATIOS (relative to |R₀|):")
print("-" * 50)
for k in range(1, 4):
    ratio = ss_abs[k] / ss_abs[0]
    print(f"  |R{k}|/|R₀| = {ratio:.6f}")

# Adjacent ratios 
print(f"\nADJACENT RATIOS:")
print("-" * 50)
for k in range(1, 4):
    ratio = ss_abs[k] / ss_abs[k-1]
    p_coupling = SA.primes[k]  # prime at this level
    print(f"  |R{k}|/|R{k-1}| = {ratio:.6f}  (p{k+1}={p_coupling})")

# Prime arithmetic search for steady-state ratios
print(f"\nPRIME ARITHMETIC MATCHES for steady-state amplitudes:")
print("=" * 70)

# Search for simple prime expressions matching the ratios
import itertools
for k in range(1, 4):
    ratio = ss_abs[k] / ss_abs[0]
    print(f"\n  |R{k}|/|R₀| = {ratio:.6f}")
    for e2,e3,e5,e7 in itertools.product(range(-3,4), repeat=4):
        cand = (2**e2)*(3**e3)*(5**e5)*(7**e7)
        if cand > 0.01 and abs(cand - ratio)/ratio < 0.02:
            parts = []
            for e,p in [(e2,2),(e3,3),(e5,5),(e7,7)]:
                if e != 0: parts.append(f"{p}^{e}" if abs(e)>1 else str(p))
            expr = "×".join(parts)
            ppm = (ratio - cand)/cand * 1e6
            frac = Fraction(cand).limit_denominator(1000)
            print(f"    {frac} = {expr} = {cand:.6f}  ({ppm:+.0f} ppm)")

# KEY: Ratio of SS amplitudes enters the cross-level via log
print(f"\n\n{'='*70}")
print(f"CROSS-LEVEL DECOMPOSITION via steady-state scaling")
print(f"{'='*70}")
A0 = np.log(rms_g1_c[0])  # transient g1
A3 = np.log(rms_g1_c[3])
D0 = np.log(rms_g2_c[0])  # steady state g2
D3 = np.log(rms_g2_c[3])
print(f"  ln(RMS_g1[R₀]) = {A0:.6f}   ln(RMS_g1[R₃]) = {A3:.6f}   Δ_trans = {A3-A0:.6f}")
print(f"  ln(RMS_g2[R₀]) = {D0:.6f}   ln(RMS_g2[R₃]) = {D3:.6f}   Δ_SS = {D3-D0:.6f}")
print(f"\n  ln(CP₀) = {A0-D0:.6f} = ln(trans₀) - ln(SS₀)")
print(f"  ln(CP₃) = {A3-D3:.6f} = ln(trans₃) - ln(SS₃)")
print(f"  Cross-level = {(A0-D0)/(A3-D3):.6f}")
print(f"\n  SS ratio = |RMS_g2[R₃]|/|RMS_g2[R₀]| = {rms_g2_c[3]/rms_g2_c[0]:.4f}")
print(f"  Trans ratio = |RMS_g1[R₃]|/|RMS_g1[R₀]| = {rms_g1_c[3]/rms_g1_c[0]:.4f}")

STEADY-STATE AMPLITUDES at ci=191
R₀: -0.01097546  |R₀| = 0.01097546  (analytical: 0.01098141)
R1: +0.02749752  |R1| = 0.02749752
R2: -0.04364412  |R2| = 0.04364412
R3: +0.27948621  |R3| = 0.27948621

SIGN PATTERN: ['-', '+', '-', '+']
  → Alternating: True

SCALING RATIOS (relative to |R₀|):
--------------------------------------------------
  |R1|/|R₀| = 2.505364
  |R2|/|R₀| = 3.976519
  |R3|/|R₀| = 25.464648

ADJACENT RATIOS:
--------------------------------------------------
  |R1|/|R0| = 2.505364  (p2=3)
  |R2|/|R1| = 1.587202  (p3=5)
  |R3|/|R2| = 6.403754  (p4=7)

PRIME ARITHMETIC MATCHES for steady-state amplitudes:

  |R1|/|R₀| = 2.505364
    2209/898 = 2^-2×3^3×5^3×7^-3 = 2.459913  (+18477 ppm)
    5/2 = 2×5 = 2.500000  (+2146 ppm)
    343/135 = 3^-3×5×7^3 = 2.540741  (-13924 ppm)
    125/49 = 5^3×7^-2 = 2.551020  (-17897 ppm)
    63/25 = 3^2×5^-2×7 = 2.520000  (-5808 ppm)

  |R2|/|R₀| = 3.976519
    875/216 = 2^-3×3^-3×5^3×7 = 4.050926  (-18368 ppm)
    225/56 = 2^-3×3^2×5^2

In [22]:
# Compact summary of SS scaling
print("SS AMPLITUDE SCALING — SUMMARY")
print("=" * 60)
for k in range(4):
    r = ss_abs[k] / ss_abs[0]
    sign = '+' if ss_vals[k] > 0 else '-'
    print(f"  R{k}: {sign}{ss_abs[k]:.8f}  |R{k}|/|R₀| = {r:.4f}")

print(f"\n  Adjacent ratios:")
for k in range(1, 4):
    r = ss_abs[k] / ss_abs[k-1]
    print(f"    R{k}/R{k-1} = {r:.6f}  (coupling prime p{k+1}={SA.primes[k]})")

print(f"\n  SS factor R₃/R₀ = {ss_abs[3]/ss_abs[0]:.4f}")
print(f"  Trans factor = {rms_g1_c[3]/rms_g1_c[0]:.4f}")
print(f"\n  Cross-level (from logs) = {(np.log(cp_q[0]))/(np.log(cp_q[3])):.6f}")
print(f"  25/9 = {25/9:.6f}")

SS AMPLITUDE SCALING — SUMMARY
  R0: -0.01097546  |R0|/|R₀| = 1.0000
  R1: +0.02749752  |R1|/|R₀| = 2.5054
  R2: -0.04364412  |R2|/|R₀| = 3.9765
  R3: +0.27948621  |R3|/|R₀| = 25.4646

  Adjacent ratios:
    R1/R0 = 2.505364  (coupling prime p2=3)
    R2/R1 = 1.587202  (coupling prime p3=5)
    R3/R2 = 6.403754  (coupling prime p4=7)

  SS factor R₃/R₀ = 25.4646
  Trans factor = 0.8896

  Cross-level (from logs) = 2.776529
  25/9 = 2.777778


In [23]:
# ── Section 5b: Full SS periodic shape via branch-mean at all crossings ──
# At each coprime crossing ci, compute the MEAN R_k (branch-independent SS) 
# and the STD (branch-dependent transient)

n_crossings = len(cis)
mean_all = np.zeros((n_crossings, n_levels))
std_all = np.zeros((n_crossings, n_levels))
rms_total = np.zeros((n_crossings, n_levels))

for idx in range(n_crossings):
    for k in range(n_levels):
        vals = np.array([res[br][idx, k] for br in all_br])
        vals_w = np.mod(vals, 2*np.pi)
        vals_w[vals_w > np.pi] -= 2*np.pi
        mean_all[idx, k] = np.mean(vals_w)
        std_all[idx, k] = np.std(vals_w)
        rms_total[idx, k] = np.sqrt(np.mean(vals_w**2))

# Show the SS periodic shape (mean across branches) for first few crossings
print("STEADY-STATE PERIODIC SHAPE (mean across branches, wrapped)")
print("=" * 80)
print(f"{'ci':<6} {'R₀_mean':<14} {'R₁_mean':<14} {'R₂_mean':<14} {'R₃_mean':<14}")
print("-" * 80)
for idx in [0, 1, 5, 10, 20, 30, 40, 43, 47]:  # sample crossings
    ci = cis[idx]
    print(f"{ci:<6} {mean_all[idx,0]:<+14.8f} {mean_all[idx,1]:<+14.8f} "
          f"{mean_all[idx,2]:<+14.8f} {mean_all[idx,3]:<+14.8f}")

# Check where the SS amplitude is largest/smallest
print(f"\n\nSS AMPLITUDE STATISTICS (|mean| over all 48 crossings)")
print("-" * 60)
for k in range(4):
    absm = np.abs(mean_all[:, k])
    print(f"  R{k}: min={absm.min():.6f} max={absm.max():.6f} "
          f"mean={absm.mean():.6f} rms={np.sqrt(np.mean(mean_all[:,k]**2)):.6f}")

# The transient std at early vs late crossings
print(f"\n\nTRANSIENT (std across branches) — early vs late crossings")
print("-" * 60)
early_idx = [i for i in range(n_crossings) if cis[i] < 30]
late_idx = [i for i in range(n_crossings) if cis[i] > 170]
for k in range(4):
    early_std = np.mean([std_all[i, k] for i in early_idx])
    late_std = np.mean([std_all[i, k] for i in late_idx])
    print(f"  R{k}: early(<30)={early_std:.6f}, late(>170)={late_std:.6f}, "
          f"ratio={early_std/late_std:.2f}")

STEADY-STATE PERIODIC SHAPE (mean across branches, wrapped)
ci     R₀_mean        R₁_mean        R₂_mean        R₃_mean       
--------------------------------------------------------------------------------
1      -0.21021187    -0.31840033    -0.70173926    -1.09384852   
11     +1.46474030    +0.37401634    -0.05064038    -0.17882262   
23     +0.63374353    +0.77332704    -0.03333986    +0.40729108   
43     +0.15119629    +0.59081413    +1.06162847    +1.69324736   
89     -0.00419860    +0.06177946    +0.03148320    -0.20461530   
137    -0.01073430    +0.02910653    -0.03946307    +0.24475850   
179    -0.01096779    +0.02755769    -0.04346813    -0.30192739   
191    -0.01097546    +0.02749752    -0.04364412    +0.27948621   
209    -0.01097969    +0.02746227    -0.04375224    -0.30233618   


SS AMPLITUDE STATISTICS (|mean| over all 48 crossings)
------------------------------------------------------------
  R0: min=0.000720 max=1.464740 mean=0.152341 rms=0.361824
  R1: min=0.

In [24]:
# Compact output: just the key statistics  
print("KEY: SS amplitude RMS across ALL crossings")
for k in range(4):
    ss_rms = np.sqrt(np.mean(mean_all[:, k]**2))
    trans_rms = np.sqrt(np.mean(std_all[:, k]**2))
    print(f"  R{k}: SS_rms={ss_rms:.8f}  Trans_rms={trans_rms:.8f}")

# Crucially: the transient std ALSO varies across crossings
# At late crossings (ci > 170), is the transient truly dead?
for k in range(4):
    late_stds = [std_all[i, k] for i in range(n_crossings) if cis[i] > 170]
    early_stds = [std_all[i, k] for i in range(n_crossings) if cis[i] < 30]
    print(f"\n  R{k}: early std avg = {np.mean(early_stds):.6f}, "
          f"late std avg = {np.mean(late_stds):.6f}")

# Check ci=191 specifically
idx_191 = int(np.where(cis == 191)[0][0])
print(f"\nAt ci=191 specifically:")
for k in range(4):
    print(f"  R{k}: mean={mean_all[idx_191,k]:+.8f}, std={std_all[idx_191,k]:.2e}")

KEY: SS amplitude RMS across ALL crossings
  R0: SS_rms=0.36182422  Trans_rms=0.36481809
  R1: SS_rms=0.35745243  Trans_rms=0.60538367
  R2: SS_rms=0.39312382  Trans_rms=0.73036940
  R3: SS_rms=0.50163717  Trans_rms=0.81004101

  R0: early std avg = 0.835277, late std avg = 0.000008

  R1: early std avg = 1.421958, late std avg = 0.000053

  R2: early std avg = 1.634865, late std avg = 0.000125

  R3: early std avg = 1.660366, late std avg = 0.000134

At ci=191 specifically:
  R0: mean=-0.01097546, std=5.93e-06
  R1: mean=+0.02749752, std=4.02e-05
  R2: mean=-0.04364412, std=9.63e-05
  R3: mean=+0.27948621, std=1.11e-04


In [25]:
# ── SS variation across coprime crossings — Phase structure ──
# Does R₀ SS really NOT vary across integer times?
# For R₀: analytical predicts R₀(n) = C₀·e^{-κn} - α ≈ -α for all n >> 1

print("R₀ SS VALUES at a sample of late crossings (should all be ≈ -α)")
print("-" * 50)
for i in range(n_crossings):
    if cis[i] > 100:
        anal = -alpha * (1 - np.exp(-KAPPA * cis[i])) - 2*np.pi * 0 * np.exp(-KAPPA * cis[i])
        # Actually for j₀=0 the mean is just the analytical value
        print(f"  ci={cis[i]:>3}: R₀_mean = {mean_all[i,0]:+.8f}  (anal: {-alpha:+.8f})")
        if cis[i] > 180: break  # just show a few

# Now check if R₁...R₃ SS varies significantly across crossings
print(f"\n\nSS OSCILLATION across crossings (R₀ is constant; do R₁-R₃ oscillate?)")
print("=" * 70)
# Plot-like: show mean_all at all 48 crossings for each level
# Using text since we can't show figures easily
for k in range(4):
    vals = mean_all[:, k]
    # Check variation
    print(f"\n  R{k}: range = [{vals.min():.6f}, {vals.max():.6f}], "
          f"peak-to-peak = {vals.max()-vals.min():.6f}")
    # Show the oscillation pattern at ~10 evenly spaced crossings
    step = n_crossings // 10
    idxs = list(range(0, n_crossings, step))
    for i in idxs:
        bar = '|' + '#' * int(abs(vals[i]) * 20 / max(abs(vals.max()), abs(vals.min())) + 0.5)
        sign = '+' if vals[i] >= 0 else '-'
        print(f"    ci={cis[i]:>3}: {sign}{abs(vals[i]):.5f} {bar}")

# KEY: At which crossings is the SS smallest/largest for each level?
print(f"\n\nCROSSING WHERE SS IS MINIMUM (most interesting for CP):")
for k in range(4):
    absm = np.abs(mean_all[:, k])
    min_idx = np.argmin(absm)
    max_idx = np.argmax(absm)
    print(f"  R{k}: min |SS| at ci={cis[min_idx]} ({absm[min_idx]:.6f}), "
          f"max |SS| at ci={cis[max_idx]} ({absm[max_idx]:.6f})")

# The quark crossings
print(f"\n  For quark g2 (ci={cis[g2_idx]}): "
      f"|SS| = [{np.abs(mean_all[g2_idx, 0]):.6f}, {np.abs(mean_all[g2_idx, 1]):.6f}, "
      f"{np.abs(mean_all[g2_idx, 2]):.6f}, {np.abs(mean_all[g2_idx, 3]):.6f}]")

R₀ SS VALUES at a sample of late crossings (should all be ≈ -α)
--------------------------------------------------
  ci=101: R₀_mean = -0.00801808  (anal: -0.01098141)
  ci=103: R₀_mean = -0.00840009  (anal: -0.01098141)
  ci=107: R₀_mean = -0.00902273  (anal: -0.01098141)
  ci=109: R₀_mean = -0.00927522  (anal: -0.01098141)
  ci=113: R₀_mean = -0.00968677  (anal: -0.01098141)
  ci=121: R₀_mean = -0.01023600  (anal: -0.01098141)
  ci=127: R₀_mean = -0.01048871  (anal: -0.01098141)
  ci=131: R₀_mean = -0.01060755  (anal: -0.01098141)
  ci=137: R₀_mean = -0.01073430  (anal: -0.01098141)
  ci=139: R₀_mean = -0.01076616  (anal: -0.01098141)
  ci=143: R₀_mean = -0.01081808  (anal: -0.01098141)
  ci=149: R₀_mean = -0.01087345  (anal: -0.01098141)
  ci=151: R₀_mean = -0.01088737  (anal: -0.01098141)
  ci=157: R₀_mean = -0.01091925  (anal: -0.01098141)
  ci=163: R₀_mean = -0.01094032  (anal: -0.01098141)
  ci=167: R₀_mean = -0.01095023  (anal: -0.01098141)
  ci=169: R₀_mean = -0.01095425  (ana

In [26]:
# Key summary: SS range and where quark crossings fall
print("SS OSCILLATION RANGE per level")
print("=" * 50)
for k in range(4):
    vals = mean_all[:, k]
    print(f"  R{k}: [{vals.min():+.6f}, {vals.max():+.6f}] p-to-p={vals.max()-vals.min():.6f}")

print(f"\nQuark g1 (ci={cis[g1_idx]}):  SS = {[f'{mean_all[g1_idx,k]:+.6f}' for k in range(4)]}")
print(f"Quark g2 (ci={cis[g2_idx]}):  SS = {[f'{mean_all[g2_idx,k]:+.6f}' for k in range(4)]}")

# The g2 SS determines the CP denominator.
# Express |SS(R3, g2)| / |SS(R0, g2)| 
r30_g2 = abs(mean_all[g2_idx, 3]) / abs(mean_all[g2_idx, 0])
print(f"\n|SS(R3)|/|SS(R0)| at g2: {r30_g2:.4f}  (compare: p3² = 25)")

# Now the KEY question: is |SS(Rk, g2)| the SAME as the total RMS_g2?
# Yes, because std_g2 ≈ 0. So rms_g2 ≈ |mean_g2|.
print(f"\nVerification: rms_g2 vs |mean_g2|")
for k in range(4):
    print(f"  R{k}: rms_g2={rms_g2_c[k]:.8f}, |mean|={abs(mean_all[g2_idx,k]):.8f}, "
          f"match={abs(rms_g2_c[k]-abs(mean_all[g2_idx,k]))<1e-6}")

# So the cross-level is determined by the SPECIFIC SS PHASE at ci=191.
# R₀ at ALL integer times = -α (constant). True?
print(f"\nR₀ SS at all late crossings (ci>100):")
late_R0 = [mean_all[i, 0] for i in range(n_crossings) if cis[i] > 100]
print(f"  All values: {[f'{v:.6f}' for v in late_R0]}")
print(f"  All ≈ -α={-alpha:.6f}? Max dev: {max(abs(v - (-alpha)) for v in late_R0):.2e}")

SS OSCILLATION RANGE per level
  R0: [-0.210212, +1.464740] p-to-p=1.674952
  R1: [-0.318400, +1.301328] p-to-p=1.619728
  R2: [-0.701739, +1.510057] p-to-p=2.211796
  R3: [-1.093849, +1.693247] p-to-p=2.787096

Quark g1 (ci=11):  SS = ['+1.464740', '+0.374016', '-0.050640', '-0.178823']
Quark g2 (ci=191):  SS = ['-0.010975', '+0.027498', '-0.043644', '+0.279486']

|SS(R3)|/|SS(R0)| at g2: 25.4646  (compare: p3² = 25)

Verification: rms_g2 vs |mean_g2|
  R0: rms_g2=0.01097546, |mean|=0.01097546, match=True
  R1: rms_g2=0.02749755, |mean|=0.02749752, match=True
  R2: rms_g2=0.04364423, |mean|=0.04364412, match=True
  R3: rms_g2=0.27948624, |mean|=0.27948621, match=True

R₀ SS at all late crossings (ci>100):
  All values: ['-0.008018', '-0.008400', '-0.009023', '-0.009275', '-0.009687', '-0.010236', '-0.010489', '-0.010608', '-0.010734', '-0.010766', '-0.010818', '-0.010873', '-0.010887', '-0.010919', '-0.010940', '-0.010950', '-0.010954', '-0.010961', '-0.010968', '-0.010970', '-0.01097

In [ ]:
## Section 6: Analytical Mechanism

The cross-level factor decomposes cleanly into transient and steady-state contributions:

$$\ln(\mathrm{CP}_k) = \ln(\mathrm{RMS}_{g1}(k)) - \ln(\mathrm{RMS}_{g2}(k)) = \ln(\text{Trans}_k) - \ln(\text{SS}_k)$$

**Transient at g1 (ci=11):**
- R₀: Binary branch structure (j₀ ∈ {0,1}). Only j₀=1 gives non-zero wrapped value ≈ 2π·e^{-κ·11}. RMS ≈ π√2·e^{-κ·ci_{g1}}
- R₃: All 7 branch values wrap to quasi-fill [-π, π]. RMS → π/√3 (uniform distribution limit)
- Trans₀/Trans₃ ≈ √6·e^{-κ·ci_{g1}} ≈ 1.15 (mild factor)

**Steady-state at g2 (ci=191):**
- R₀: Analytically derived. |SS₀| = α = εω/(ω²+κ²) ≈ 1/(2π√P₄)
- R₃: Cascade amplification. |SS₃| ≈ p₃²·α (≈ 2% systematic deviation)
- SS₃/SS₀ ≈ p₃² = 25 (dominant factor)

**The cross-level factor is dominated by SS scaling.** The 25× amplification of SS from R₀ to R₃ creates most of the 25/9 ratio. The transient compression (÷1.15) and the log structure together produce the exact ratio.

In [27]:
# ── Section 6: Verify analytical predictions vs measured ──

decay_g1 = np.exp(-KAPPA * cis[g1_idx])
print(f"ci_g1 = {cis[g1_idx]}, decay = e^(-κ·{cis[g1_idx]}) = {decay_g1:.6f}")

# ── TRANSIENT at g1 ──
# R₀: binary structure, RMS ≈ π√2·decay (from j₀ ∈ {0,1})
trans_R0_pred = np.pi * np.sqrt(2) * decay_g1
print(f"\nTRANSIENT (g1 = ci={cis[g1_idx]})")
print(f"  R₀: predicted π√2·decay = {trans_R0_pred:.6f}, measured = {rms_g1_c[0]:.6f}, "
      f"dev = {(rms_g1_c[0]-trans_R0_pred)/trans_R0_pred*100:+.2f}%")

# R₃: quasi-uniform wrapping, RMS → π/√3
trans_R3_pred = np.pi / np.sqrt(3)
print(f"  R₃: predicted π/√3 = {trans_R3_pred:.6f}, measured = {rms_g1_c[3]:.6f}, "
      f"dev = {(rms_g1_c[3]-trans_R3_pred)/trans_R3_pred*100:+.2f}%")

# Trans ratio
trans_ratio_pred = np.sqrt(6) * decay_g1
trans_ratio_meas = rms_g1_c[0] / rms_g1_c[3]
print(f"  Trans₀/Trans₃: predicted √6·decay = {trans_ratio_pred:.6f}, "
      f"measured = {trans_ratio_meas:.6f}, dev = {(trans_ratio_meas-trans_ratio_pred)/trans_ratio_pred*100:+.2f}%")

# ── STEADY STATE at g2 ──
# R₀: analytical = α
print(f"\nSTEADY STATE (g2 = ci={cis[g2_idx]})")
print(f"  R₀: predicted α = {alpha:.8f}, measured = {rms_g2_c[0]:.8f}, "
      f"dev = {(rms_g2_c[0]-alpha)/alpha*100:+.2f}%")

# R₃: cascade amplification ≈ p₃²·α
ss_R3_pred = 25 * alpha  # p₃² · α
print(f"  R₃: predicted p₃²α = {ss_R3_pred:.8f}, measured = {rms_g2_c[3]:.8f}, "
      f"dev = {(rms_g2_c[3]-ss_R3_pred)/ss_R3_pred*100:+.2f}%")

# SS ratio
print(f"  SS₃/SS₀: predicted p₃² = {25}, measured = {rms_g2_c[3]/rms_g2_c[0]:.4f}, "
      f"dev = {(rms_g2_c[3]/rms_g2_c[0]-25)/25*100:+.2f}%")

# ── CROSS-LEVEL from these approximations ──
print(f"\nCROSS-LEVEL FACTOR")
ln_cp0_pred = np.log(trans_R0_pred / alpha)
ln_cp3_pred = np.log(trans_R3_pred / (25*alpha))
cross_pred = ln_cp0_pred / ln_cp3_pred
print(f"  Predicted: ln(π√2·decay/α) / ln((π/√3)/(p₃²α))")
print(f"  = ln({trans_R0_pred/alpha:.2f}) / ln({trans_R3_pred/(25*alpha):.2f})")
print(f"  = {ln_cp0_pred:.6f} / {ln_cp3_pred:.6f}")
print(f"  = {cross_pred:.6f}")
print(f"  Target: 25/9 = {25/9:.6f}")
print(f"  Deviation: {(cross_pred - 25/9)/(25/9)*100:+.3f}%")

# Simplify: the predicted cross-level is
# ln(π√2·e^{-κ·ci} · (ω²+κ²)/(εω)) / ln(π/(√3 · p₃²) · (ω²+κ²)/(εω))
# = ln(π√2·e^{-κ·ci}/α) / ln(π/(√3·p₃²·α))
# Both numerator and denominator depend on α. Let β = 1/α = (ω²+κ²)/(εω) ≈ ω/ε
beta = 1/alpha
print(f"\n  β = 1/α = {beta:.4f}")
print(f"  ω/ε = {OMEGA/EPSILON:.4f}")
print(f"  2π√P₄ = {2*np.pi*np.sqrt(SA.P):.4f}")
print(f"\n  Cross-level = ln(π√2·decay·β) / ln(π·β/(√3·p₃²))")

# If β >> 1 (which it is: β ≈ 91), then ln(π√2·decay·β) ≈ ln(β) + ln(π√2·decay)
# and ln(π·β/(√3·p₃²)) ≈ ln(β) + ln(π/(√3·p₃²))
# So cross ≈ (ln β + A) / (ln β + B) where A,B are O(1) constants
A = np.log(np.pi * np.sqrt(2) * decay_g1)
B = np.log(np.pi / (np.sqrt(3) * 25))
print(f"\n  A = ln(π√2·decay) = {A:.6f}")
print(f"  B = ln(π/(√3·p₃²)) = {B:.6f}")
print(f"  ln(β) = {np.log(beta):.6f}")
print(f"  cross ≈ (ln β + A)/(ln β + B) = ({np.log(beta):.4f} + {A:.4f})/({np.log(beta):.4f} + {B:.4f})")
print(f"        = {(np.log(beta)+A)/(np.log(beta)+B):.6f}")

ci_g1 = 11, decay = e^(-κ·11) = 0.468101

TRANSIENT (g1 = ci=11)
  R₀: predicted π√2·decay = 2.079716, measured = 2.075590, dev = -0.20%
  R₃: predicted π/√3 = 1.813799, measured = 1.846494, dev = +1.80%
  Trans₀/Trans₃: predicted √6·decay = 1.146608, measured = 1.124071, dev = -1.97%

STEADY STATE (g2 = ci=191)
  R₀: predicted α = 0.01098141, measured = 0.01097546, dev = -0.05%
  R₃: predicted p₃²α = 0.27453525, measured = 0.27948624, dev = +1.80%
  SS₃/SS₀: predicted p₃² = 25, measured = 25.4646, dev = +1.86%

CROSS-LEVEL FACTOR
  Predicted: ln(π√2·decay/α) / ln((π/√3)/(p₃²α))
  = ln(189.39) / ln(6.61)
  = 5.243783 / 1.888099
  = 2.777281
  Target: 25/9 = 2.777778
  Deviation: -0.018%

  β = 1/α = 91.0630
  ω/ε = 91.0520
  2π√P₄ = 91.0520

  Cross-level = ln(π√2·decay·β) / ln(π·β/(√3·p₃²))

  A = ln(π√2·decay) = 0.732231
  B = ln(π/(√3·p₃²)) = -2.623452
  ln(β) = 4.511551
  cross ≈ (ln β + A)/(ln β + B) = (4.5116 + 0.7322)/(4.5116 + -2.6235)
        = 2.777281


In [ ]:
## Section 7: Summary and Verdict

### Mechanism Identified

The cross-level factor 25/9 in x_q = (4/7)(25/9) arises from the **differential scaling of transient and steady-state amplitudes** between the innermost (R₀) and outermost (R₃) cascade levels at the quark crossing pair (ci=11, ci=191):

$$\frac{\ln(\text{CP}_0)}{\ln(\text{CP}_3)} = \frac{\ln(\text{Trans}_0) - \ln(\text{SS}_0)}{\ln(\text{Trans}_3) - \ln(\text{SS}_3)} = \frac{\ln\beta + A}{\ln\beta + B}$$

where:
- β = 2π√P₄ ≈ 91.1 (inverse of R₀ steady-state, derived analytically)
- A = ln(π√2·e^{-κ·ci_{g1}}) ≈ 0.73 (R₀ transient: binary branch wrapping)
- B = ln(π/(√3·p₃²)) ≈ -2.62 (R₃ transient ÷ SS amplification)

### Three Physical Components

1. **R₀ transient** ≈ π√2·e^{-κ·11} — binary branch (j₀ ∈ {0,1}), large-deviation statistics. Verified to 0.2%.

2. **R₃ transient** ≈ π/√3 — seven-fold branch (j₃ ∈ {0,...,6}) wraps quasi-uniformly in [-π,π]. Verified to 1.8%.

3. **SS amplification** |SS₃|/|SS₀| ≈ p₃² = 25 — the cascade transfer amplifies the R₃ steady-state by approximately p₃² relative to R₀. Verified to 1.9%.

### Cross-Level Accuracy

| Component | Predicted | Measured | Deviation |
|-----------|-----------|----------|-----------|
| Trans(R₀) | π√2·decay | 2.076 | -0.20% |
| Trans(R₃) | π/√3 | 1.847 | +1.80% |
| SS(R₃)/SS(R₀) | p₃² = 25 | 25.46 | +1.86% |
| **Cross-level** | **2.7773** | **2.7765** | **-0.018%** |

The individual components are each ~2% off, but **the log structure absorbs these deviations**: errors in the transient and SS partially cancel, yielding 25/9 to 0.018%.

### What Remains Open

The **SS amplification law** |SS₃|/|SS₀| ≈ p₃² is measured, not derived. Proving that the cascade transfer function produces exactly p₃² amplification at the quark crossing time would close GAP-20 completely. This requires solving the nonlinear cascade transfer in steady state — analytically intractable for now, but the target is now precisely identified.

In [28]:
# ── Scorecard ──
print("NB171 SCORECARD")
print("=" * 65)
print("""
0 new identities (investigation/mechanism).

FINDINGS:
1. Cross-level factor 25/9 decomposed into 3 physical components:
   - R₀ transient: π√2·e^(-κ·ci_g1) (binary wrapping, 0.2% match)
   - R₃ transient: π/√3 (quasi-uniform wrapping, 1.8% match)
   - SS amplification: |SS₃|/|SS₀| ≈ p₃² = 25 (1.9% match)

2. Analytical formula: cross = (ln β + A)/(ln β + B)
   where β = 2π√P₄, reproduces 25/9 to 0.018% (180 ppm)

3. R₀ ODE is exactly solvable: R₀(t) decoupled, analytical at
   integer times. Verified against scipy and JAX.

4. Subsystem cascade approach FAILED — generic crossing selection
   doesn't capture channel-specific structure. 16-min scipy run
   was a dead end.

5. The remaining open question reduced from "why 25/9?" to
   "why does SS₃/SS₀ = p₃² in the nonlinear cascade?"

GAP-20 STATUS: FURTHER NARROWED
  x_q = 100/63 = (4/7)(25/9) confirmed (NB170)
  4/7 = x(R₀) DERIVED (NB161)
  25/9 MECHANISM IDENTIFIED (NB171) -- individual components ~2% off,
  but log structure gives 0.018%. SS amplification law ≈ p₃² is the
  remaining analytical target.

Running total: 281 predictions/identities, 0 free parameters
""")
print(f"Verification: cross-level formula = {(np.log(1/alpha) + np.log(np.pi*np.sqrt(2)*decay_g1))/(np.log(1/alpha) + np.log(np.pi/(np.sqrt(3)*25))):.6f}")
print(f"Target 25/9 = {25/9:.6f}")

NB171 SCORECARD

0 new identities (investigation/mechanism).

FINDINGS:
1. Cross-level factor 25/9 decomposed into 3 physical components:
   - R₀ transient: π√2·e^(-κ·ci_g1) (binary wrapping, 0.2% match)
   - R₃ transient: π/√3 (quasi-uniform wrapping, 1.8% match)
   - SS amplification: |SS₃|/|SS₀| ≈ p₃² = 25 (1.9% match)

2. Analytical formula: cross = (ln β + A)/(ln β + B)
   where β = 2π√P₄, reproduces 25/9 to 0.018% (180 ppm)

3. R₀ ODE is exactly solvable: R₀(t) decoupled, analytical at
   integer times. Verified against scipy and JAX.

4. Subsystem cascade approach FAILED — generic crossing selection
   doesn't capture channel-specific structure. 16-min scipy run
   was a dead end.

5. The remaining open question reduced from "why 25/9?" to
   "why does SS₃/SS₀ = p₃² in the nonlinear cascade?"

GAP-20 STATUS: FURTHER NARROWED
  x_q = 100/63 = (4/7)(25/9) confirmed (NB170)
  4/7 = x(R₀) DERIVED (NB161)
  25/9 MECHANISM IDENTIFIED (NB171) -- individual components ~2% off,
  but log